# Traning the self built GPT2 model from scratch

In [ ]:
# Config
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

num_return_sequences = 5
max_length = 30

torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Model
from nanoGPT2 import GPT, GPTConfig

# model = GPT.from_pretrained("gpt2")
model = GPT(GPTConfig())
model.eval()
model.to(device)

print(device)

In [ ]:
# Load data
with open('training_data/shakespeare.txt', 'r') as f:
    text = f.read()

data = text[:1000]
print(data[:100])
# # Encode with tiktoken gpt2 bpe
# enc = tiktoken.get_encoding("gpt2")
# tokens = enc.encode(text)


In [ ]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode(data)
print(tokens[:24])

In [ ]:
B,T = 4,32

# This will create 4 batches of 32 tokens each (each row is a batch).
# The model will predict a token for each batch.
# x: [4, 32] is the input to the model.
# y: [4, 32] is the target to the model.
# The output/target is shifted by 1 token to the right, so the model will predict the next token in the sequence.
buf = torch.tensor(tokens[:B*T + 1])
x = buf[:-1].view(B, T).to(device)
y = buf[1:].view(B, T).to(device)
print(x[:4,:6])
print(y[:4,:6])

# Now the the expected output in y is at the same position as the input in x.
# So for example for the third index (token 25), the expected output is y's third index (token 198).

In [ ]:
# Testing forward pass
logits, _ = model(x)
print(logits.shape)

In [ ]:
# calculate loss
logits, loss = model(x, y)
print("loss: ", loss)

# Expected loss is -ln(1/50257) (cross entropy loss for random initialization)
import math
print("expected loss: ", -math.log(1/50257))

In [ ]:
# Training on single batch of data for verification
max_iters = 50
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
for iter in range(max_iters):
    optimizer.zero_grad()
    logits, loss = model(x, y)
    loss.backward()
    optimizer.step()
    print(f"step {iter} loss: {loss.item()}")

# This test shows that we can overfit a single batch of data, which is a good sign and expected.

In [ ]:
# import importlib
# import nanoGPT2
# importlib.reload(nanoGPT2)  # pick up edits to nanoGPT2.py without restarting the kernel
from nanoGPT2 import DataLoaderTest

# Training loop
train_loader = DataLoaderTest(B, T)

for iter in range(max_iters):
    x, y = train_loader.next_batch()
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad(set_to_none=True)
    logits, loss = model(x, y)
    loss.backward()
    optimizer.step()
    print(f"step {iter} loss: {loss.item()}")
